# Notebook 01 — ML Baseline (Người 1)

**Chủ đề:** Phân tích cảm xúc tiếng Việt bằng **Machine Learning truyền thống**.

**Mục tiêu:** train + đánh giá 3 model cổ điển trên UIT-VSFC.

| Model | Vector hoá | Mục tiêu |
|---|---|---|
| Naive Bayes | TF-IDF | baseline nhanh |
| Logistic Regression | TF-IDF | baseline tuyến tính |
| SVM (LinearSVC) | TF-IDF | baseline cạnh tranh |

**Yêu cầu trước:** chạy `00_data_preparation.ipynb` để có `data/processed/*.csv`.

**Đầu ra cần có:**
- Bảng so sánh accuracy / precision / recall / F1 / confusion matrix
- File `models/ml_baseline.pkl` (best model)
- Mô tả ngắn về điểm mạnh / yếu

In [ ]:
# --- Imports & paths --------------------------------------------------------
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'
MODELS = ROOT / 'models'; MODELS.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / 'results'; RESULTS.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
NUM_CLASSES = 3

In [ ]:
# Load data
def load_split(name):
    df = pd.read_csv(PROC / f'{name}.csv')
    df['text_clean'] = df['text_clean'].fillna(' ').astype(str)
    df['label'] = df['label'].astype(int)
    return df

df_train = load_split('train')
df_dev   = load_split('dev')
df_test  = load_split('test')
print(df_train.shape, df_dev.shape, df_test.shape)

## 1. Hàm đánh giá chung (Người 1 nên giữ cùng format với Người 2 & 3)

In [ ]:
def evaluate(y_true, y_pred, name, show_cm=True):
    """In metrics + vẽ confusion matrix. Trả về dict cho bảng so sánh."""
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    print(f'\n=== {name} ===')
    print(f'Accuracy {acc:.4f} | Precision {p:.4f} | Recall {r:.4f} | F1 macro {f1:.4f}')
    print(classification_report(y_true, y_pred,
                                target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)],
                                zero_division=0))
    if show_cm:
        cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
        plt.figure(figsize=(4, 3))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=[LABEL_NAMES[i] for i in range(NUM_CLASSES)],
                    yticklabels=[LABEL_NAMES[i] for i in range(NUM_CLASSES)], cbar=False)
        plt.title(f'{name}'); plt.xlabel('Predicted'); plt.ylabel('True')
        plt.tight_layout(); plt.show()
    return {'model': name, 'accuracy': acc, 'precision': p, 'recall': r, 'f1_macro': f1}

## 2. Train 3 model — TODO

Mỗi model là 1 `sklearn.Pipeline` (vectorizer + classifier). Người 1 có thể thử thêm 
**BoW** (CountVectorizer) bên cạnh **TF-IDF** để có nhận xét cho báo cáo.

In [ ]:
def make_pipe(clf, vec='tfidf'):
    """Tạo pipeline vectorize + classifier. Cho phép swap nhanh giữa BoW và TF-IDF."""
    if vec == 'bow':
        v = CountVectorizer(ngram_range=(1, 2), min_df=2, max_features=30_000)
    else:
        v = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30_000, sublinear_tf=True)
    return Pipeline([('vec', v), ('clf', clf)])

models = {
    'Naive Bayes (TF-IDF)':         make_pipe(MultinomialNB()),
    'Logistic Regression (TF-IDF)': make_pipe(LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)),
    'SVM (TF-IDF)':                 make_pipe(LinearSVC(C=1.0, class_weight='balanced')),
}

results = []
trained = {}
for name, pipe in models.items():
    print(f'Training {name} ...')
    pipe.fit(df_train['text_clean'], df_train['label'])
    y_pred = pipe.predict(df_test['text_clean'])
    results.append(evaluate(df_test['label'].values, y_pred, name, show_cm=False))
    trained[name] = pipe

In [ ]:
# Bảng so sánh + lưu model tốt nhất theo F1 macro
ml_results = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
print(ml_results)

best_name = ml_results.iloc[0]['model']
joblib.dump(trained[best_name], MODELS / 'ml_baseline.pkl')
ml_results.to_csv(RESULTS / 'ml_baseline.csv', index=False)
print(f'\n[saved] best={best_name} -> models/ml_baseline.pkl')

## 3. TODO của Người 1

- [ ] Thử **BoW vs TF-IDF** (đổi `vec='bow'`) và so sánh.
- [ ] Tune `C` cho SVM/LR bằng GridSearchCV trên dev.
- [ ] Vẽ confusion matrix cho **best model** (đặt `show_cm=True`).
- [ ] Thêm phần nhận xét: **điểm mạnh** (nhanh, ít tài nguyên) / **điểm yếu** (không hiểu ngữ cảnh, phụ thuộc nặng vào tiền xử lý).
- [ ] Đảm bảo metric format giống Người 2 & 3 để dễ làm bảng tổng hợp cuối báo cáo.